In [2]:
import os, re, time
import ee
import geemap
import folium
from osgeo import gdal, ogr, osr


workDir = os.getcwd()
print("Current working directory:", workDir)
gridGPKG =  workDir + "/data/wildE/_00_SHAPEFILES/wildE_Grid_GLANCE_EU_150km.gpkg"
DL_folder = workDir + "/data/_PREDICTIONS/"
if not os.path.exists(DL_folder):
    os.makedirs(DL_folder)


# Earth Engine initialisieren
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()


Current working directory: \\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\thesis\scripts\01_exploration



Successfully saved authorization token.


In [3]:
# Create an interactive map
Map = geemap.Map(center=[40, -5], zoom=6)
Map.addLayerControl()


# Add a basemap
Map.add_basemap('CartoDB.DarkMatter')

# ESA FireCCI51 burned area
firecci51 = ee.ImageCollection('ESA/CCI/FireCCI/5_1').filterDate('2010-01-01', '2010-12-31')
cci_burnedArea = firecci51.select('BurnDate')
maxBA = cci_burnedArea.max()
baVis = {
    'min': 1,
    'max': 366,
    'palette': [
        'ff0000', 'fd4100', 'fb8200', 'f9c400', 'f2ff00', 'b6ff05',
        '7aff0a', '3eff0f', '02ff15', '00ff55', '00ff99', '00ffdd',
        '00ddff', '0098ff', '0052ff', '0210ff', '3a0dfb', '7209f6',
        'a905f1', 'e102ed', 'ff00cc', 'ff0089', 'ff0047', 'ff0004'
    ]
}
Map.addLayer(maxBA, baVis, 'FireCCI_max_2010')


# MODIS Burned Area (MCD64A1.061)
modis = ee.ImageCollection('MODIS/061/MCD64A1').filter(ee.Filter.date('2010-01-01', '2010-12-31'))
modis_burnedArea = modis.select('BurnDate')
modis_burnedAreaVis = {
    'min': 30.0,
    'max': 341.0,
    'palette': ['4e0400', '951003', 'c61503', 'ff1901'],
}
Map.addLayer(modis_burnedArea, modis_burnedAreaVis, 'MBA_2010')

# Global Fire Atlas (GFA) 2010
gfa_ignitions_2010 = ee.FeatureCollection("projects/sat-io/open-datasets/global-fire-atlas/ignitions/Global_fire_atlas_V1_ignitions_2010")
gfa_perimeter_2010 = ee.FeatureCollection("projects/sat-io/open-datasets/global-fire-atlas/perimeter/Global_fire_atlas_V1_perimeter_2010")

Map.addLayer(gfa_ignitions_2010, {}, 'GFA_Ignitions 2010')
Map.addLayer(gfa_perimeter_2010, {}, 'GFA_Perimeter 2010')

# GABAM
gabam = ee.ImageCollection("projects/sat-io/open-datasets/GABAM")
y_2010 = gabam.filterDate('2010-01-01', '2010-12-31')

Map.addLayer(y_2010.mosaic(), {'min': 1, 'max': 1, 'palette': "f00c0c"}, 'GABAM 2010')

# Globfire
globfirePerimeters = ee.FeatureCollection("JRC/GWIS/GlobFire/v2/FinalPerimeters")

# Filter to a specific time period (e.g., 2010 fires)
fires2010 = globfirePerimeters.filter(ee.Filter.And(
    ee.Filter.gte('IDate', ee.Date('2010-01-01').millis()),
    ee.Filter.lte('FDate', ee.Date('2010-12-31').millis()))
)

# Basic visualization
Map.addLayer(fires2010, {'color': 'red'}, 'Globfire 2010')

# Add example polygons 
poi_1 = ee.Geometry.Polygon(
    [[[-8.477308994205861, 41.04250160279352],
      [-8.477308994205861, 40.9512909743108],
      [-8.304617648991018, 40.9512909743108],
      [-8.304617648991018, 41.04250160279352]]], None)

poi_2 = ee.Geometry.Polygon(
    [[[-0.9588744040777453, 39.09185862500175],
      [-0.9588744040777453, 38.707105927420876],
      [-0.14039295876524527, 38.707105927420876],
      [-0.14039295876524527, 39.09185862500175]]], None)

poi_3 = ee.Geometry.Polygon(
    [[[-10.058336022721333, 53.53805676251327],
      [-10.058336022721333, 53.26705463193047],
      [-9.297532800065083, 53.26705463193047],
      [-9.297532800065083, 53.53805676251327]]], None)

poi_4 = ee.Geometry.Polygon(
    [[[19.369899832621368, 55.39082778842778],
      [19.369899832621368, 54.15842969105259],
      [23.154689871683868, 54.15842969105259],
      [23.154689871683868, 55.39082778842778]]], None)

# Add the polygons to the map
Map.addLayer(poi_1, {'color': 'blue'}, 'Polygon 1')
Map.addLayer(poi_2, {'color': 'green'}, 'Polygon 2')
Map.addLayer(poi_3, {'color': 'orange'}, 'Polygon 3')
Map.addLayer(poi_4, {'color': 'purple'}, 'Polygon 4')


# safe map as html
Map.save('fire_exploration.html')

# Display the map
Map

Map(center=[40, -5], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chil…